# Gather OOD probe corporaDownloads AMPERSAND (Reddit CMV), PERSUADE 2.0 (student essays), and AURC (UKP web arguments) for the out-of-distribution probes in Section 6 of the paper.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p phase2_data/raw/ukp_sentential
cd phase2_data/raw/ukp_sentential

# Try HuggingFace first
python3 <<'PY'
from datasets import load_dataset
import json
candidates = [
    ("UKPLab/UKP_ASPECT",         "arg_mining"),
    ("UKPLab/UKPSententialArgMin", None),
    ("stab-corpus-2018",          None),
    ("UKP_ASPECT",                None),
]
for name, subset in candidates:
    try:
        ds = load_dataset(name, subset) if subset else load_dataset(name)
        print(f"OK: {name}  splits: {list(ds.keys())}  first_keys: {list(next(iter(ds[list(ds.keys())[0]])).keys())}")
        break
    except Exception as e:
        print(f"FAIL {name}: {str(e)[:100]}")
PY

# If HF failed, download from GitHub
echo ""
echo "=== falling back to GitHub direct download ==="
wget -q https://raw.githubusercontent.com/UKPLab/acl2018-project-argument-mining/master/data/ukp_sentential/arg-quality-rank-30k.txt 2>&1 | head -5 || true

# Also try the Trautmann AURC dataset (backup)
wget -q --show-progress -O aurc_data.zip \
  "https://raw.githubusercontent.com/trtm/AURC/master/data/AURC_DATA.zip" 2>&1 | tail -5 || true

ls -la

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p phase2_data/raw/aurc phase2_data/raw/persuade
cd phase2_data/raw/aurc

# ── Approach 1: git clone the Trautmann AURC repo (known-good source) ──
echo "=== [1] git clone AURC repo ==="
git clone --depth 1 https://github.com/trtm/AURC.git 2>&1 | tail -5
ls -la AURC/data/ 2>/dev/null | head -20

# ── Approach 2: search HuggingFace for what's actually available ──
echo ""
echo "=== [2] HuggingFace search for arg-mining datasets ==="
python3 <<'PY'
try:
    from huggingface_hub import HfApi
    api = HfApi()
    print("--- search: 'argument mining' ---")
    for d in list(api.list_datasets(search="argument mining", limit=15)):
        print(f"  {d.id}")
    print("\n--- search: 'claim detection' ---")
    for d in list(api.list_datasets(search="claim detection", limit=10)):
        print(f"  {d.id}")
    print("\n--- search: 'persuade' ---")
    for d in list(api.list_datasets(search="persuade", limit=10)):
        print(f"  {d.id}")
    print("\n--- search: 'persuasive essays' ---")
    for d in list(api.list_datasets(search="persuasive essays", limit=10)):
        print(f"  {d.id}")
    print("\n--- search: 'kialo' ---")
    for d in list(api.list_datasets(search="kialo", limit=5)):
        print(f"  {d.id}")
except Exception as e:
    print(f"HF search failed: {e}")
PY

# ── Approach 3: try downloading Chakrabarty CMV corpus (GitHub, known URL) ──
echo ""
echo "=== [3] Chakrabarty CMV corpus (github) ==="
cd ~/argument-aware-rag/phase2_data/raw
git clone --depth 1 https://github.com/tuhinjubcse/AMPERSAND-EMNLP2019.git 2>&1 | tail -5
ls -la AMPERSAND-EMNLP2019/ 2>/dev/null | head -10

In [ ]:
%%bash
cd ~/argument-aware-rag/phase2_data/raw

echo "=== AURC ==="
echo "--- AURC_DATA.tsv (first 3 lines) ---"
head -3 aurc/AURC/data/AURC_DATA.tsv 2>/dev/null || head -3 aurc/AURC_DATA.tsv
echo ""
echo "--- unique labels in AURC_DATA.tsv col 2/3/4 ---"
awk -F'\t' 'NR>1 {print $2}' aurc/AURC/data/AURC_DATA.tsv 2>/dev/null | sort -u | head
awk -F'\t' 'NR>1 {print $3}' aurc/AURC/data/AURC_DATA.tsv 2>/dev/null | sort -u | head
awk -F'\t' 'NR>1 {print $4}' aurc/AURC/data/AURC_DATA.tsv 2>/dev/null | sort -u | head
echo ""
echo "--- row count ---"
wc -l aurc/AURC/data/AURC_DATA.tsv

echo ""
echo "=== AMPERSAND ==="
echo "--- claimtrain.tsv (first 3 lines) ---"
head -3 AMPERSAND-EMNLP2019/claimtrain.tsv
echo ""
echo "--- claimdev.tsv (first 3 lines) ---"
head -3 AMPERSAND-EMNLP2019/claimdev.tsv
echo ""
echo "--- argmining/ contents ---"
ls -la AMPERSAND-EMNLP2019/argmining/
echo ""
echo "--- unique labels in claimtrain col 2 ---"
awk -F'\t' '{print $2}' AMPERSAND-EMNLP2019/claimtrain.tsv | sort -u | head

In [ ]:
%%bash
cd ~/argument-aware-rag/phase2_data/raw

echo "=== AURC_SENTENCE_LEVEL_STANCE.tsv ==="
head -5 aurc/AURC/data/AURC_SENTENCE_LEVEL_STANCE.tsv
echo ""
echo "row count:"; wc -l aurc/AURC/data/AURC_SENTENCE_LEVEL_STANCE.tsv
echo ""
echo "unique stance labels (col 3 or 4):"
awk -F'\t' 'NR>1 {print $3}' aurc/AURC/data/AURC_SENTENCE_LEVEL_STANCE.tsv | sort -u
awk -F'\t' 'NR>1 {print $4}' aurc/AURC/data/AURC_SENTENCE_LEVEL_STANCE.tsv | sort -u
echo ""
echo "unique topics:"
awk -F'\t' 'NR>1 {print $1}' aurc/AURC/data/AURC_SENTENCE_LEVEL_STANCE.tsv | sort -u

echo ""
echo "=== AMPERSAND README ==="
cat AMPERSAND-EMNLP2019/README.md

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p eval_logs

# ── First: sanity-check the stance label distribution and topics ──
python3 <<'PY'
import pandas as pd
from pathlib import Path
data = pd.read_csv('phase2_data/raw/aurc/AURC/data/AURC_DATA.tsv', sep='\t')
stance = pd.read_csv('phase2_data/raw/aurc/AURC/data/AURC_SENTENCE_LEVEL_STANCE.tsv', sep='\t')
print(f"AURC_DATA: {len(data)} rows, cols: {list(data.columns)[:10]}")
print(f"AURC_STANCE: {len(stance)} rows, cols: {list(stance.columns)}")
print("\nstance label distribution:")
print(stance['sentence_level_stance'].value_counts())
print("\ntopic distribution (data):")
print(data['topic'].value_counts())
PY
